In [1]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

### Data loading ###

In [2]:
df = pd.read_csv("../datasets/groceries.csv")
df.head()

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10
0,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,NaN
1,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,NaN
4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,NaN


### Inspecting the data ###

In [3]:
df.shape

(9835, 10)

In [4]:
df.columns.to_list()

['Item 1',
 'Item 2',
 'Item 3',
 'Item 4',
 'Item 5',
 'Item 6',
 'Item 7',
 'Item 8',
 'Item 9',
 'Item 10']

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9835 entries, 0 to 9834
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Item 1   9835 non-null   str  
 1   Item 2   7676 non-null   str  
 2   Item 3   6033 non-null   str  
 3   Item 4   4734 non-null   str  
 4   Item 5   3729 non-null   str  
 5   Item 6   2874 non-null   str  
 6   Item 7   2229 non-null   str  
 7   Item 8   1684 non-null   str  
 8   Item 9   1246 non-null   str  
 9   Item 10  896 non-null    str  
dtypes: str(10)
memory usage: 768.5 KB


### Data cleaning ###

In [6]:
for column in df.columns:
    df[column] = df[column].fillna("")
    df[column] = df[column].astype(str)
    df[column] = df[column].str.strip()
    df[column] = df[column].str.lower()

df

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10
0,citrus fruit,semi-finished bread,margarine,ready soups,,,,,,
1,tropical fruit,yogurt,coffee,,,,,,,
2,whole milk,,,,,,,,,
3,pip fruit,yogurt,cream cheese,meat spreads,,,,,,
4,other vegetables,whole milk,condensed milk,long life bakery product,,,,,,
...,...,...,...,...,...,...,...,...,...,...
9830,sausage,chicken,beef,hamburger meat,citrus fruit,grapes,root vegetables,whole milk,butter,whipped/sour cream
9831,cooking chocolate,,,,,,,,,
9832,chicken,citrus fruit,other vegetables,butter,yogurt,frozen dessert,domestic eggs,rolls/buns,rum,cling film/bags
9833,semi-finished bread,bottled water,soda,bottled beer,,,,,,


### Converting the row data into transaction ###

We will make something like below
<!-- transactions = [
    ['Bread', 'Milk'],
    ['Bread', 'Butter', 'Milk'],
    ['Bread', 'Butter'],
    ['Milk', 'Butter'],
    ['Bread', 'Milk', 'Butter', 'Jam'],
    ['Bread', 'Milk', 'Butter'],
    ['Milk', 'Butter', 'Jam'],
    ['Bread', 'Butter']
] -->


In [7]:
transactions = []

for index, row in df.iterrows():
    basket = []
    for item in row:
        if item != "":
            if item not in basket:
                basket.append(item)

    if len(basket)> 0:
        transactions.append(basket)

#transactions

### Converting transaction into boolean table ###

In [8]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

basket_df = pd.DataFrame(
    te_array,
    columns=te.columns_
)
basket_df.sample(5)

,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,...,uht-milk,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
9442,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3063,False,False,False,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
9372,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
7484,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1864,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Apply PF-Growth ###

In [9]:
frequent_itemsets = fpgrowth(
    basket_df,
    min_support= 0.01,
    use_colnames=True
)

print(frequent_itemsets)


      support                                          itemsets
0    0.082766                         frozenset({citrus fruit})
1    0.054906                            frozenset({margarine})
2    0.016268                  frozenset({semi-finished bread})
3    0.138892                               frozenset({yogurt})
4    0.104931                       frozenset({tropical fruit})
..        ...                                               ...
256  0.016167  frozenset({other vegetables, frozen vegetables})
257  0.010371   frozenset({frozen vegetables, root vegetables})
258  0.011083            frozenset({frozen vegetables, yogurt})
259  0.014235             frozenset({other vegetables, onions})
260  0.011998                   frozenset({whole milk, onions})

[261 rows x 2 columns]


In [10]:
# Sorting by support
frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

print("Frequent itemsets:")
print(frequent_itemsets.head(20).to_string(index=False))

Frequent itemsets:
 support                                  itemsets
0.255313                   frozenset({whole milk})
0.193493             frozenset({other vegetables})
0.180580                   frozenset({rolls/buns})
0.164108                         frozenset({soda})
0.138892                       frozenset({yogurt})
0.108998              frozenset({root vegetables})
0.104931               frozenset({tropical fruit})
0.103305                frozenset({bottled water})
0.093950                      frozenset({sausage})
0.085003                frozenset({shopping bags})
0.084087                       frozenset({pastry})
0.082766                 frozenset({citrus fruit})
0.075648                    frozenset({pip fruit})
0.075445                  frozenset({canned beer})
0.075038                 frozenset({bottled beer})
0.074631 frozenset({other vegetables, whole milk})
0.070564           frozenset({whipped/sour cream})
0.069243                   frozenset({newspapers})
0.062227    

### Generate association rule ###

In [11]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.30
)

rules = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

# Convert frozensets to readable text
rules["antecedents"] = rules["antecedents"].apply(
lambda items: ", ".join(sorted(items))
)

rules["consequents"] = rules["consequents"].apply(
lambda items: ", ".join(sorted(items))
)


# Sort the rules by lift
rules = rules.sort_values(
by="lift",
ascending=False
)

print("\nAssociation rules:")
print(rules.head(20).to_string(index=False))


Association rules:
                     antecedents      consequents  support  confidence     lift
  citrus fruit, other vegetables  root vegetables 0.010371    0.359155 3.295045
other vegetables, tropical fruit  root vegetables 0.012303    0.342776 3.144780
                            beef  root vegetables 0.017387    0.331395 3.040367
   citrus fruit, root vegetables other vegetables 0.010371    0.586207 3.029608
 root vegetables, tropical fruit other vegetables 0.012303    0.584541 3.020999
    other vegetables, whole milk  root vegetables 0.022979    0.307902 2.824828
         root vegetables, yogurt other vegetables 0.012506    0.493976 2.552944
      tropical fruit, whole milk           yogurt 0.014642    0.347826 2.504297
     rolls/buns, root vegetables other vegetables 0.010676    0.481651 2.489249
     root vegetables, whole milk other vegetables 0.022979    0.471816 2.438420
                          onions other vegetables 0.014235    0.459016 2.372268
                pork

### Explanations ###
Support: 0.010, about 1% of shoping transaction contains all three prodcuts - citrus fruit, other vegetables and root vegetables \
Confidence: Who bought citrus fruit and other vegetables, about 35.9% of them also bought root vegetables. \
Lift: 3.29 which is greater than 1. A strong relationship.